# Token Sequence Workbench

Run tokens one by one, inspect `State` after each step, and prototype new tokens.

---
## Part 1 — Environment Setup

In [ ]:
import os, sys

%cd /content

# ── Clone repos ──
if not os.path.exists("graph_Time_series"):
    !git clone https://github.com/chahineNejm/graph_Time_series
if not os.path.exists("kernels_playground"):
    !git clone https://github.com/chahineNejm/kernels_playground

# ── Make importable ──
for p in ["/content/graph_Time_series",
          "/content/kernels_playground",
          "/content/kernels_playground/first_tests"]:
    if p not in sys.path:
        sys.path.insert(0, p)

!pip install -q datasets

print("Ready.")

---
## Part 2 — Load Token Framework

In [ ]:
from pathlib import Path
import importlib.util
import types
import numpy as np

PACKAGE_DIR = Path("/content/graph_Time_series/graph_Time_series")


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


# Register parent packages so submodule imports resolve
pkg = types.ModuleType("graph_Time_series")
pkg.__path__ = [str(PACKAGE_DIR)]
sys.modules.setdefault("graph_Time_series", pkg)

blocks = types.ModuleType("graph_Time_series.token_blocks")
blocks.__path__ = [str(PACKAGE_DIR / "token_blocks")]
sys.modules.setdefault("graph_Time_series.token_blocks", blocks)

# Core modules
state_mod = load_module("graph_Time_series.state", PACKAGE_DIR / "state.py")
token_mod = load_module("graph_Time_series.token", PACKAGE_DIR / "token.py")
norm_mod  = load_module("graph_Time_series.token_blocks.normalization",
                        PACKAGE_DIR / "token_blocks" / "normalization.py")
rbf_mod   = load_module("graph_Time_series.token_blocks.kernel_rbf",
                        PACKAGE_DIR / "token_blocks" / "kernel_rbf.py")

State                = state_mod.State
TransformRecord      = state_mod.TransformRecord
ZNormalizationToken  = norm_mod.ZNormalizationToken
KernelRBFToken       = rbf_mod.KernelRBFToken

print("Loaded from", PACKAGE_DIR)

---
## Part 3 — Load Data

Uses `kernels_playground` utilities (same pattern as `tweaks.ipynb`).

In [ ]:
from utils.config import DATASETS
from utils.data import build_examples

MAX_SAMPLES = 32   # 24 run + 8 holdout
HOLDOUT     = 8

raw = build_examples(
    config="electricity_H_long",
    start=0, stop=100, step=1,
    dataset_name=DATASETS["eval"],
)
print(f"Loaded {len(raw)} raw examples")

# Stack into 2-D arrays for State(H, F)
examples = raw[:MAX_SAMPLES]
min_hist = min(len(e["history"]) for e in examples)
min_fut  = min(len(e["future"])  for e in examples)

H_all = np.stack([e["history"][-min_hist:] for e in examples]).astype(np.float32)
F_all = np.stack([e["future"][:min_fut]    for e in examples]).astype(np.float32)

H, F             = H_all[:-HOLDOUT], F_all[:-HOLDOUT]
H_holdout, F_holdout = H_all[-HOLDOUT:], F_all[-HOLDOUT:]

print(f"Run data:      H {H.shape}  F {F.shape}")
print(f"Held-out data: H {H_holdout.shape}  F {F_holdout.shape}")

---
## Part 4 — State Inspector

`inspect(state)` gives you a full snapshot after any token.
`diff(before, after)` highlights what changed.

In [ ]:
import textwrap

def inspect(state, label="State"):
    """Pretty-print everything interesting in a State object."""
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  tokens applied : {state.token_sequence}")
    print(f"  class counts   : {state.class_counts}")
    print(f"  terminated     : {state.terminated}")
    print()

    # Shapes
    print("  Arrays:")
    print(f"    original_history   {state.original_history.shape}")
    print(f"    original_future    {state.original_future.shape}")
    print(f"    active_target_base {state.active_target_base.shape}")
    print(f"    current_target     {state.current_target.shape}")
    print()

    # Features
    def _feat(d):
        if not d:
            return "{}"
        return ", ".join(f"{k} {v.shape}" for k, v in d.items())
    print(f"  historical_features: {_feat(state.historical_features)}")
    print(f"  future_features:     {_feat(state.future_features)}")
    print()

    # Transforms
    if state.transform_stack:
        print("  Transform stack:")
        for i, t in enumerate(state.transform_stack):
            inv = "yes" if t.inverse_fn else "no"
            print(f"    [{i}] {t.name}  (inverse: {inv}, affects: {t.affects})")
            if t.params:
                for k, v in t.params.items():
                    vstr = f"{v}" if not hasattr(v, "shape") else f"array {v.shape}"
                    print(f"         {k}: {vstr}")
    else:
        print("  Transform stack: (empty)")
    print()

    # Predictions
    if state.prediction_stack:
        print(f"  Prediction stack ({len(state.prediction_stack)}):")
        for i, (name, pred) in enumerate(zip(state.prediction_names, state.prediction_stack)):
            rng = f"[{pred.min():.4f}, {pred.max():.4f}]"
            print(f"    [{i}] {name:20s} {pred.shape}  range {rng}")
    else:
        print("  Prediction stack: (empty)")

    # Flags
    if state.flags:
        print(f"  Flags: {state.flags}")
    print(f"{'=' * 60}\n")


def diff(before, after, label=""):
    """Show what changed between two State snapshots."""
    title = f"Diff: {label}" if label else "Diff"
    print(f"\n{'─' * 50}")
    print(f"  {title}")
    print(f"{'─' * 50}")

    # New tokens
    new_tokens = after.token_sequence[len(before.token_sequence):]
    if new_tokens:
        print(f"  + tokens: {new_tokens}")

    # New features
    for store_name in ("historical_features", "future_features"):
        old_keys = set(getattr(before, store_name).keys())
        new_keys = set(getattr(after, store_name).keys())
        added = new_keys - old_keys
        if added:
            feats = getattr(after, store_name)
            for k in sorted(added):
                print(f"  + {store_name}.{k}  {feats[k].shape}")

    # New transforms
    n_old = len(before.transform_stack)
    for t in after.transform_stack[n_old:]:
        print(f"  + transform: {t.name}  (affects: {t.affects})")

    # New predictions
    n_old_pred = len(before.prediction_stack)
    for i, (name, pred) in enumerate(
        zip(after.prediction_names[n_old_pred:], after.prediction_stack[n_old_pred:])
    ):
        print(f"  + prediction: {name}  {pred.shape}")

    # Target change
    if not np.array_equal(before.active_target_base, after.active_target_base):
        print(f"  ~ active_target_base changed")
    if not np.array_equal(before.current_target, after.current_target):
        resid_norm = np.linalg.norm(after.current_target)
        print(f"  ~ current_target changed  (||residual|| = {resid_norm:.4f})")

    # New flags
    new_flags = {k: v for k, v in after.flags.items() if k not in before.flags}
    if new_flags:
        print(f"  + flags: {new_flags}")

    print(f"{'─' * 50}\n")


print("inspect() and diff() ready.")

---
## Part 5 — Token Registry

Register all available tokens here.  Add your own at the bottom.

In [ ]:
# ── Built-in tokens ──
TOKENS = {
    "ZNormalization": ZNormalizationToken(),
    "kernel_rbf":     KernelRBFToken(),
}


# ── Add your custom tokens below ──
# from graph_Time_series.token_blocks.my_token import MyToken
# TOKENS["my_token"] = MyToken()


print("Registered tokens:", list(TOKENS.keys()))

---
## Part 6 — Define & Run Sequence

Edit `TOKEN_SEQUENCE` to change the pipeline. The runner applies each
token, snapshots the state, and shows `inspect()` + `diff()` after
every step.

In [ ]:
TOKEN_SEQUENCE = [
    "ZNormalization",
    "kernel_rbf",
]

In [ ]:
# ── Step-by-step runner with full inspection ──
state = State(H, F)
snapshots = {"init": state.copy()}
inspect(state, "INIT")

for i, name in enumerate(TOKEN_SEQUENCE):
    token = TOKENS[name]
    print(f"\n{'#' * 60}")
    print(f"  Step {i+1}: applying  {name}")
    print(f"  can_apply = {token.can_apply(state)}")
    print(f"{'#' * 60}")

    before = state.copy()
    state = token.apply(state)
    snapshots[name] = state.copy()

    inspect(state, f"after {name}")
    diff(before, state, label=name)

print("\nAll snapshots stored in `snapshots` dict.")
print("Keys:", list(snapshots.keys()))

---
## Part 7 — Final Forecast

In [ ]:
forecast = state.get_final_prediction()
print("Final forecast shape:", forecast.shape)
print("Sample 0 (first 10):", np.round(forecast[0, :10], 2))

---
## Part 8 — Execution Log

In [ ]:
state.print_log()

---
## Part 9 — Re-inspect Any Snapshot

Change the key to look at state after any token.

In [ ]:
# Pick any snapshot: "init", "ZNormalization", "kernel_rbf", ...
inspect(snapshots["ZNormalization"], "snapshot: ZNormalization")

In [ ]:
# Compare any two snapshots
diff(snapshots["init"], snapshots["ZNormalization"], label="init → ZNormalization")

---
---
## Part 10 — New Token Scratch Space

Use this area to prototype new tokens. A token needs:
- `can_apply(state) -> bool`
- `apply(state) -> State`

Minimal template below. Copy it, rename, and register in Part 5.

In [ ]:
# ── Token template ──
# Uncomment and edit to create your own token.

# class MyNewToken:
#     """One-line description of what this token does."""
#
#     TOKEN_NAME  = "my_new_token"
#     TOKEN_CLASS = "transform"  # or "model", "feature", ...
#
#     def can_apply(self, state):
#         # Return True if this token is valid in the current state.
#         # Example: only apply if no normalization has been done yet.
#         return "normalized" not in state.flags
#
#     def apply(self, state):
#         state = state.copy()
#
#         # ── Do your work here ──
#         # Example: log-transform the target
#         # offset = 1.0
#         # transformed = np.log(state.active_target_base + offset)
#         # state.register_transform(
#         #     name=self.TOKEN_NAME,
#         #     inverse_fn=lambda pred: np.exp(pred) - offset,
#         #     transformed_target=transformed,
#         #     params={"offset": offset},
#         # )
#
#         # ── Record the token ──
#         state.record_token(self.TOKEN_NAME, self.TOKEN_CLASS)
#         return state

In [ ]:
# ── Quick test: run your new token in isolation ──

# my_token = MyNewToken()
# test_state = State(H[:4], F[:4])   # small batch for speed
# print("can_apply:", my_token.can_apply(test_state))
# result = my_token.apply(test_state)
# inspect(result, "after MyNewToken")

In [ ]:
# ── Once it works, register it and re-run Part 6 ──

# TOKENS["my_new_token"] = MyNewToken()
# TOKEN_SEQUENCE.append("my_new_token")
# # Then re-run Part 6 cells above